In [ ]:
import numpy as np
import pandas as pd
from sklearn import tree

In [13]:
df = pd.read_csv("/content/drive/MyDrive/Linear Regression/student-mat.csv")
df

,weekly_studytime,failures,extra_edu_supp,family_edu_supp,Interested_in_higher_edu,internet_access,freetime_after_school,goout_with_friends,absences,G1,G2,G3
0,2,0,yes,no,yes,no,3,4,6,5,6,6
1,2,0,no,yes,yes,yes,3,3,4,5,5,6
2,2,3,yes,no,yes,yes,3,2,10,7,8,10
3,3,0,no,yes,yes,yes,2,2,2,15,14,15
4,2,0,no,yes,yes,no,3,2,4,6,10,10
...,...,...,...,...,...,...,...,...,...,...,...,...
390,2,2,no,yes,yes,no,5,4,11,9,9,9
391,1,0,no,no,yes,yes,4,5,3,14,16,16
392,1,3,no,no,yes,no,5,3,3,10,8,7
393,1,0,no,no,yes,yes,4,1,0,11,12,10


In [14]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['weekly_studytime', 'failures', 'extra_edu_supp', 'family_edu_supp', 'Interested_in_higher_edu', 'internet_access', 'freetime_after_school', 'goout_with_friends', 'absences', 'G1', 'G2', 'G3']


In [15]:
df['extra_edu_supp'] = df['extra_edu_supp'].str.strip()
df['family_edu_supp'] = df['family_edu_supp'].str.strip()
df['Interested_in_higher_edu'] = df['Interested_in_higher_edu'].str.strip()

In [16]:
print(df['extra_edu_supp'].unique())
print(df['family_edu_supp'].unique())
print(df['Interested_in_higher_edu'].unique())

['yes' 'no']
['no' 'yes']
['yes' 'no']


In [17]:
df['pass_fail'] = df['G3'].apply(
    lambda x: "Pass" if x >= 10 else "Fail"
)

In [18]:
df[['G3','pass_fail']].head()

,G3,pass_fail
0,6,Fail
1,6,Fail
2,10,Pass
3,15,Pass
4,10,Pass


In [19]:
from sklearn.preprocessing import LabelEncoder

le_extra = LabelEncoder()
le_family = LabelEncoder()
le_higher = LabelEncoder()
le_result = LabelEncoder()

df['extra_edu_supp'] = le_extra.fit_transform(df['extra_edu_supp'])

df['family_edu_supp'] = le_family.fit_transform(df['family_edu_supp'])

df['Interested_in_higher_edu'] = le_higher.fit_transform(df['Interested_in_higher_edu'])

df['pass_fail'] = le_result.fit_transform(df['pass_fail'])

In [20]:
print(le_extra.classes_)
print(le_family.classes_)
print(le_higher.classes_)
print(le_result.classes_)

['no' 'yes']
['no' 'yes']
['no' 'yes']
['Fail' 'Pass']


In [24]:
df['internet_access'] = df['internet_access'].str.strip()

le_internet = LabelEncoder()

df['internet_access'] = le_internet.fit_transform(
    df['internet_access']
)

In [25]:
print(le_internet.classes_)

['no' 'yes']


In [26]:
X = df[[
    'weekly_studytime',
    'failures',
    'extra_edu_supp',
    'family_edu_supp',
    'Interested_in_higher_edu',
    'internet_access',
    'freetime_after_school',
    'goout_with_friends',
    'absences',
    'G1',
    'G2'
]]

y = df['pass_fail']

In [27]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
dt = tree.DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt.fit(x_train, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

In [29]:
training_accuracy = dt.score(x_train, y_train)
testing_accuracy = dt.score(x_test, y_test)

print("Training Accuracy:", training_accuracy * 100, "%")
print("Testing Accuracy:", testing_accuracy * 100, "%")

Training Accuracy: 95.56962025316456 %
Testing Accuracy: 87.34177215189874 %


In [30]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = dt.predict(x_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le_result.classes_
))

Confusion Matrix:
[[25  1]
 [ 9 44]]

Classification Report:
              precision    recall  f1-score   support

        Fail       0.74      0.96      0.83        26
        Pass       0.98      0.83      0.90        53

    accuracy                           0.87        79
   macro avg       0.86      0.90      0.87        79
weighted avg       0.90      0.87      0.88        79



In [31]:
test_student = pd.DataFrame([{
    "weekly_studytime": 3,
    "failures": 0,
    "extra_edu_supp": le_extra.transform(["yes"])[0],
    "family_edu_supp": le_family.transform(["yes"])[0],
    "Interested_in_higher_edu": le_higher.transform(["yes"])[0],
    "internet_access": le_internet.transform(["yes"])[0],
    "freetime_after_school": 3,
    "goout_with_friends": 2,
    "absences": 4,
    "G1": 12,
    "G2": 14
}])

prediction = dt.predict(test_student)

result = le_result.inverse_transform(prediction)

print("Student Result:", result[0])

Student Result: Pass


In [32]:
probabilities = dt.predict_proba(test_student)[0]

classes = list(le_result.classes_)

fail_index = classes.index("Fail")
pass_index = classes.index("Pass")

print("Fail Probability:", probabilities[fail_index] * 100, "%")
print("Pass Probability:", probabilities[pass_index] * 100, "%")

Fail Probability: 0.0 %
Pass Probability: 100.0 %


In [33]:
import joblib

joblib.dump(dt, "student_performance_model.pkl")
joblib.dump(le_extra, "extra_support_encoder.pkl")
joblib.dump(le_family, "family_support_encoder.pkl")
joblib.dump(le_higher, "higher_education_encoder.pkl")
joblib.dump(le_internet, "internet_access_encoder.pkl")
joblib.dump(le_result, "student_result_encoder.pkl")

['student_result_encoder.pkl']

In [34]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

@st.cache_resource
def load_files():
    model = joblib.load("student_performance_model.pkl")
    extra_encoder = joblib.load("extra_support_encoder.pkl")
    family_encoder = joblib.load("family_support_encoder.pkl")
    higher_encoder = joblib.load("higher_education_encoder.pkl")
    internet_encoder = joblib.load("internet_access_encoder.pkl")
    result_encoder = joblib.load("student_result_encoder.pkl")

    return (
        model,
        extra_encoder,
        family_encoder,
        higher_encoder,
        internet_encoder,
        result_encoder
    )

(
    model,
    extra_encoder,
    family_encoder,
    higher_encoder,
    internet_encoder,
    result_encoder
) = load_files()

st.set_page_config(
    page_title="Student Performance Predictor",
    page_icon="🎓",
    layout="centered"
)

st.title("🎓 Student Performance Prediction System")

st.write(
    "Enter the student's academic and lifestyle details "
    "to predict whether the student is likely to pass or fail."
)

with st.form("student_form"):

    weekly_studytime = st.number_input(
        "Weekly Study Time",
        min_value=1,
        max_value=4,
        value=2,
        step=1
    )

    failures = st.number_input(
        "Previous Failures",
        min_value=0,
        max_value=4,
        value=0,
        step=1
    )

    extra_edu_supp = st.selectbox(
        "Extra Educational Support",
        extra_encoder.classes_.tolist()
    )

    family_edu_supp = st.selectbox(
        "Family Educational Support",
        family_encoder.classes_.tolist()
    )

    interested_higher = st.selectbox(
        "Interested in Higher Education",
        higher_encoder.classes_.tolist()
    )

    internet_access = st.selectbox(
        "Internet Access",
        internet_encoder.classes_.tolist()
    )

    freetime_after_school = st.number_input(
        "Free Time After School",
        min_value=1,
        max_value=5,
        value=3,
        step=1
    )

    goout_with_friends = st.number_input(
        "Going Out With Friends",
        min_value=1,
        max_value=5,
        value=3,
        step=1
    )

    absences = st.number_input(
        "Absences",
        min_value=0,
        value=0,
        step=1
    )

    G1 = st.number_input(
        "First Period Grade (G1)",
        min_value=0,
        max_value=20,
        value=10,
        step=1
    )

    G2 = st.number_input(
        "Second Period Grade (G2)",
        min_value=0,
        max_value=20,
        value=10,
        step=1
    )

    submitted = st.form_submit_button("Predict Student Result")

if submitted:

    try:
        input_data = pd.DataFrame([{
            "weekly_studytime": weekly_studytime,
            "failures": failures,
            "extra_edu_supp": extra_encoder.transform(
                [extra_edu_supp]
            )[0],
            "family_edu_supp": family_encoder.transform(
                [family_edu_supp]
            )[0],
            "Interested_in_higher_edu": higher_encoder.transform(
                [interested_higher]
            )[0],
            "internet_access": internet_encoder.transform(
                [internet_access]
            )[0],
            "freetime_after_school": freetime_after_school,
            "goout_with_friends": goout_with_friends,
            "absences": absences,
            "G1": G1,
            "G2": G2
        }])

        prediction = model.predict(input_data)
        probabilities = model.predict_proba(input_data)[0]

        result = result_encoder.inverse_transform(
            prediction
        )[0]

        classes = list(result_encoder.classes_)
        fail_index = classes.index("Fail")
        pass_index = classes.index("Pass")

        if result == "Pass":
            st.success(f"Student Result: {result}")
        else:
            st.error(f"Student Result: {result}")

        st.write(
            f"Pass Probability: "
            f"{probabilities[pass_index] * 100:.2f}%"
        )

        st.write(
            f"Fail Probability: "
            f"{probabilities[fail_index] * 100:.2f}%"
        )

    except Exception as error:
        st.error("Prediction could not be completed.")
        st.exception(error)

Writing app.py


In [35]:
%%writefile requirements.txt
streamlit
pandas
joblib
scikit-learn==1.6.1

Writing requirements.txt


In [37]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")
files.download("student_performance_model.pkl")
files.download("extra_support_encoder.pkl")
files.download("family_support_encoder.pkl")
files.download("higher_education_encoder.pkl")
files.download("internet_access_encoder.pkl")
files.download("student_result_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>